# 🤖 JARVIS Fine-Tuning — 100+ Examples
**Runtime: GPU T4 (free) | ~35 min**

In [ ]:
!pip install unsloth transformers datasets trl peft bitsandbytes -q
print('✅ Done')

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Phi-3.5-mini-instruct',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print('✅ Model loaded')

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('✅ LoRA ready')

In [ ]:
# Upload jarvis_training_data.py first then run this
from google.colab import files
print('Upload jarvis_training_data.py from your JARVIS/colab/ folder')
uploaded = files.upload()

In [ ]:
import importlib, jarvis_training_data as jd
from datasets import Dataset

SYSTEM = jd.JARVIS_SYSTEM
CONVERSATIONS = jd.CONVERSATIONS
print(f'Loaded {len(CONVERSATIONS)} training examples')

def fmt(c):
    msgs = [
        {'role':'system',    'content': SYSTEM},
        {'role':'user',      'content': c['user']},
        {'role':'assistant', 'content': c['jarvis']},
    ]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list([fmt(c) for c in CONVERSATIONS])
print(f'✅ Dataset ready: {len(dataset)} examples')

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=4,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        output_dir='/content/jarvis-checkpoints',
        report_to='none',
    ),
)
print('🚀 Training...')
trainer.train()
print('✅ Done!')

In [ ]:
FastLanguageModel.for_inference(model)

def ask(q):
    msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':q}]
    inp = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=inp, max_new_tokens=300, temperature=0.72, top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True).strip()

for q in ['Hey JARVIS how are you?','Explain machine learning simply','I am stressed about my deadline','Are you better than ChatGPT?','What is the meaning of life?']:
    print(f'YOU: {q}')
    print(f'JARVIS: {ask(q)}')
    print()

In [ ]:
model.save_pretrained_gguf('/content/jarvis-model', tokenizer, quantization_method='q4_k_m')
print('✅ GGUF saved!')
import os
for f in os.listdir('/content/jarvis-model'):
    print(f'  {f}: {os.path.getsize("/content/jarvis-model/"+f)/1e9:.2f} GB')

In [ ]:
# Download to your PC
from google.colab import files
import glob
for f in glob.glob('/content/jarvis-model/*.gguf'):
    print(f'Downloading {f}...')
    files.download(f)
    print('✅ Save this .gguf file to your JARVIS project folder')
    break